***

## Preparing Workspace

***

In [24]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft


In [25]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git     = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

<>:8: SyntaxWarning: invalid escape sequence '\R'
<>:8: SyntaxWarning: invalid escape sequence '\R'
C:\Users\jchoy\AppData\Local\Temp\ipykernel_8468\2084991863.py:8: SyntaxWarning: invalid escape sequence '\R'
  path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')


In [26]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

***

## Preparing Imports

***

In [50]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying ACS data
indicator_name     = df_params[df_params['Type'] == 'indicator_name' ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'       ]['Input'].values[0]
sample_type        = df_params[df_params['Type'] == 'sample'         ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'      ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'     ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'    ]['Input'].values[0]
margin_of_error    = df_params[df_params['Type'] == 'margin_of_error']['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'       ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'     ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'       ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'   ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'      ]['Input'].values[0]


# view
print(indicator_name)
print(estimate)
print(sample_type)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Margin of error: " + margin_of_error)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

# Import about table
df_about = pd.read_excel(os.path.join(path_config0, 'About Indicators.xlsx'), sheet_name = 'Indicators')
df_about = df_about[df_about['Indicator'] == indicator_name]
folder = df_about.Folder.values[0]
MOE_thresh = df_about['MOE Threshold'].values[0]
print(folder)
print('MOE threshold: ' + str(MOE_thresh) + '%')

Accessibility_3
ACS5
PUMS
PUMA
PUMA
Percentages: Yes
Margin of error: Yes
Number of variables: 3
2016
2022
Auto Ownership
MOE threshold: 5%


In [51]:
## Import Variable Mapping
df_inputs = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = import_tab)

# Set years
years_to_import = list(range(year_start, year_end+1))

## For DEC data
if estimate == 'DEC':

    # Reset years to import for DEC
    # Set DEC variables to import
    # Import County FIPS mapping
    # Convert to dictionary object for easy state-county combination importing

    df_vars = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = estimate)
    df_vars = df_vars[df_vars['Indicator Name'].str.contains(indicator_name).replace(np.nan, False)]
    df_vars = df_vars[df_vars['Include'] == 'Yes']
    
    if margin_of_error == 'Yes':
        df_vars['ID_Attributes'] = df_vars['ID_Attributes'].apply(ME_split)

    years_to_import = [2000, 2010, 2020]
    
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = ['NAME'] + unique(df_vars[df_vars['Year'] == year]['ID'].to_list())
        
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'CountyFIPS'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
    df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)]
    dict_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values))
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = dict_fips[['State FIPS', 'County FIPS']]
    dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])
    
    # view
    print(dict_fips)
    print(dict_vars)

## For ACS1 or ACS5 data
if sample_type in ['ACS', 'SUBJECT']:

    df_vars = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = sample_type)
    df_vars = df_vars[df_vars['Indicator Name'].str.contains(indicator_name).replace(np.nan, False)]
    df_vars = df_vars[df_vars['Include'] == 'Yes']
    
    # Remove 2020 if pulling ACS1 tables (Census did not take an ACS1 sample in 2020)
    # Set tables and variables to import    

    if estimate == 'ACS1':
        try:
            years_to_import.remove(2020)
        except Exception as e: print(e)

    if margin_of_error == 'Yes':
        df_vars['ID_Attributes'] = df_vars['ID_Attributes'].apply(ME_split)
        list_vars = ['NAME'] + df_vars['ID_Attributes'].to_list()
    else:
        list_vars = ['NAME'] + df_vars['ID'].to_list()

    if sample_type == 'ACS':
        tables = df_vars['Table'].unique()
        print(tables)

    # For tract and county level pull
    if import_tab == 'Counties':
        
        # Import County FIPS mapping
        # Convert to dictionary object for easy state-county combination importing
        df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                                , sheet_name = 'CountyFIPS'
                                , dtype = {'State FIPS': object, 'County FIPS': object})
        
        df_fips = df_fips[
                        (df_fips['State'].isin(df_inputs['states'].values))
                        & (df_fips['County Name'].isin(df_inputs['counties'].values))
        ]
        dict_fips = df_fips.copy()
        dict_fips = dict_fips[['State FIPS', 'County FIPS']]
        dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
        
        for key in list(dict_fips.keys()):
            dict_fips[key] = ",".join(dict_fips[key])
    
        # view
        print(dict_fips)
        print(list_vars)

    
    # For MSA level pull
    if import_tab == 'MSA':
    
        # Set MSAs to import
        df_inputs['msa'] = df_inputs['msa'].astype("string")
        msa_to_import = df_inputs['msa'].values
        msa_to_import = ",".join(msa_to_import)
    
        # view
        print(tables)
        print(msa_to_import)
        print(list_vars)


## For PUMS data
if sample_type == 'PUMS':

    # Remove 2012-2015 if pulling PUMS tables (they only reported at the state level for PUMS on these years)
    # Create dictionary of variable mappings by year (sometimes the variable name changes over time)
    # Import County FIPS mapping
    # Convert to dictionary object for easy state-county combination importing

    df_vars = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = sample_type)
    df_vars = df_vars[df_vars['Year'].isin(years_to_import)]
    df_vars = df_vars[df_vars['Indicator Name'].str.contains(indicator_name).replace(np.nan, False)]
    df_vars = df_vars[df_vars['Include'] == 'Yes']
    table_type = df_inputs['table'].values[0]
    if table_type == 'P':
        weight = 'PWGTP'
    if table_type == 'H':
        weight = 'WGTP'
    groups  = list(df_vars[df_vars['Data Type'].str.contains('group')]['ID2'].unique())
    groups2 = list(df_vars[df_vars['Data Type'] == 'group']['ID2'].unique())
    
    if estimate == 'ACS5':
        try:
            years_to_import.remove(2012)
            years_to_import.remove(2013)
            years_to_import.remove(2014)
            years_to_import.remove(2015)
        except Exception as e: print(e)

    df_vars = df_vars[df_vars['Year'].isin(years_to_import)]
            
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = unique(df_vars[(df_vars['Year'] == year) & (df_vars['Data Type'].str.contains('group'))]['ID'].to_list()) + unique(df_vars[(df_vars['Year'] == year) & (df_vars['Data Type'] == 'integer')]['ID'].to_list()) + [weight]
        
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'CountyFIPS'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
    df_fips_pums = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                                 , sheet_name = 'PUMAcodes'
                                 , dtype = {'STATEFP': object, 'COUNTYFP': object, 'TRACTCE': object, 'PUMA5CE': object})
    df_fips_pums = df_fips_pums.rename(columns = {'STATEFP':'State FIPS', 'COUNTYFP':'County FIPS'})

    df_fips = df_fips.merge(df_fips_pums[['State FIPS', 'County FIPS', 'PUMA5CE']].drop_duplicates(), on = ['State FIPS', 'County FIPS'])
    df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)]
    dict_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values))
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = dict_fips[['State FIPS', 'PUMA5CE']].drop_duplicates()
    
    dict_fips = dict_fips.groupby('State FIPS')['PUMA5CE'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])

    # view
    print(dict_fips)
    print(dict_vars)



if estimate == 'CPS':
    
    df_vars = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = sample_type)
    df_vars = df_vars[df_vars['Year'].isin(years_to_import)]
    df_vars = df_vars[df_vars['Indicator Name'].str.contains(indicator_name).replace(np.nan, False)]
    df_vars = df_vars[df_vars['Include'] == 'Yes']
    
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = unique(df_vars[df_vars['Year'] == year]['ID'].to_list()) + [df_vars[df_vars['Year'] == year]['Suggested Weight'].values[0]]
    
    # Import County FIPS mapping
    # Convert to dictionary object for easy state-county combination importing
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'CountyFIPS'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
    
    df_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values))
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = df_fips.copy()
    dict_fips = dict_fips[['State FIPS', 'County FIPS']]
    dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])
        
    # view
    print(dict_fips)
    print(dict_vars)


# view
df_vars.head(3)

C:\Users\jchoy\AppData\Local\Temp\ipykernel_8468\3135243380.py:120: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_vars = df_vars[df_vars['Indicator Name'].str.contains(indicator_name).replace(np.nan, False)]


list.remove(x): x not in list
{'06': '01700,06103,06102,06101,06707,06714,06709,06704,06703,06706,06705,06712,06701,06717,06713,06716,06715,06702,06708,06710,06711,10100,11301,11302,11300'}
{'2016': ['VEH', 'WGTP'], '2017': ['VEH', 'WGTP'], '2018': ['VEH', 'WGTP'], '2019': ['VEH', 'WGTP'], '2020': ['VEH', 'WGTP'], '2021': ['VEH', 'WGTP'], '2022': ['VEH', 'WGTP']}


,Label,ID,Value1,Value2,Description,Suggested Weight,Year,Indicator Name,ID2,Description2,Include,Data Type,Table Type
76001,Vehicles (1 ton or less) available,VEH,0,0,No vehicles,WGTP,2022,Accessibility_3,VEH,No vehicles,Yes,group,H
76002,Vehicles (1 ton or less) available,VEH,1,1,1 vehicle,WGTP,2022,Accessibility_3,VEH,1 vehicle,Yes,group,H
76003,Vehicles (1 ton or less) available,VEH,2,2,2 vehicles,WGTP,2022,Accessibility_3,VEH,2 vehicles,Yes,group,H


***

## Importing

***

In [38]:
# df_census = df_census_raw.copy()
# df_census = df_census[(df_census['PERRP'].isin(['40', '41'])) | (df_census['PERRP'].isin(['1', '2']))]

# df1 = df_census.copy()
# df2 = df_census.drop_duplicates(['HRHHID', 'HRHHID2', 'Year'])

# print(df1.shape)
# print(df2.shape)

In [52]:
start_time = time.time()

# Import Census Bureau data to url mapping
df_urls = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = 'URL')

## For ACS tables
if sample_type == 'ACS':
  
    ## Import data and concatenate onto ID fields ##
    print("Importing and compiling ACS data from the Census Bureau...")
    print("")
    
    # initialize empty list to store data frames
    # import multiple years and counties
    # iterate through each table and import all variables needed from each table
    # combine all years and counties (or MSAs)
    # reduce all tables/variables pulled into one table

    list_df_census = []
    
    for table in tables:
    
        print("")
        print("Table ID: " + table)
        print("")
        list_df_tables = []
    
        df_table = df_vars[df_vars['Table'] == table]
        if margin_of_error == 'Yes':
            list_table_vars = [['NAME'] + df_table['ID_Attributes'].to_list()[x:x+20] for x in range(0, len(df_table['ID_Attributes'].to_list()), 20)]
        else:
            list_table_vars = [['NAME'] + df_table['ID'].to_list()[x:x+45] for x in range(0, len(df_table['ID'].to_list()), 45)]
        
        list_variables = []
        for x in list_table_vars:
            list_variables.append(",".join(x))
        
        list_df_vars = []
        
        for variables in list_variables:
            print("Variables: " + variables)
            list_df_years = []

            if import_tab == 'Counties':
                for state in list(dict_fips.keys()):
                    print('State: ' + state)
                    for year in tqdm(years_to_import):
                        try:
                            list_df_years.append(
                                query_census(df_urls      = df_urls
                                              , api_key   = api_key
                                              , estimate  = estimate
                                              , sample    = sample_type
                                              , geography = geography
                                              , variables = variables
                                              , year      = year
                                              , state     = state
                                              , county    = dict_fips[state])
                            )
                        except Exception as e: print(e)
                df_years = pd.concat(list_df_years)
                            
            if import_tab == 'MSA':
                for year in tqdm(years_to_import):
                    try:
                        list_df_years.append(
                            query_census(df_urls      = df_urls
                                          , api_key   = api_key
                                          , estimate  = estimate
                                          , sample    = sample_type
                                          , geography = geography
                                          , variables = variables
                                          , year      = year
                                          , msa       = msa_to_import)
                        )
                    except Exception as e: print(e)
                df_years = pd.concat(list_df_years)

            list_df_vars.append(df_years)
        
        if geography == 'Tracts':
            df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'tract', 'Year'], how = 'outer'), list_df_vars)
        if geography == 'Counties':
            df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'Year'], how = 'outer'), list_df_vars)
        if geography == 'MSA':
            df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year'], how = 'outer'), list_df_vars)

        list_df_census.append(df_vars_all)
        print("All variables from table ID " + table + " have been reduced together into one table")
        print("")

    print("")
    print("Reducing all tables together into one final table...")
    print("")
    
    if geography == 'Tracts':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'tract', 'Year']), list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'tract', 'Year']).reset_index()
    if geography == 'Counties':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'Year']), list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'Year']).reset_index()
    if geography == 'MSA':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']), list_df_census)
        df_census_raw = df_census_raw.set_index(['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']).reset_index()



## For SUBJECT tables

if sample_type == 'SUBJECT':
    ## Import data and concatenate onto ID fields ##
    print("Importing and compiling ACS Subject data from the Census Bureau...")
    print("")
    
    # initialize empty list to store data frames
    # import multiple years and counties
    # iterate through each table and import all variables needed from each table
    # combine all years and counties (or MSAs)
    # reduce all tables/variables pulled into one table

    list_df_census = []
    if margin_of_error == 'Yes':
        list_table_vars = [['NAME'] + df_vars['ID_Attributes'].to_list()[x:x+20] for x in range(0, len(df_vars['ID_Attributes'].to_list()), 20)]
    else:
        list_table_vars = [['NAME'] + df_vars['ID'].to_list()[x:x+45] for x in range(0, len(df_vars['ID'].to_list()), 45)]
    
    list_variables = []
    for x in list_table_vars:
        list_variables.append(",".join(x))
    
    list_df_vars = []
    
    for variables in list_variables:
        print("Variables: " + variables)
        list_df_years = []
        if import_tab == 'Counties':
            for state in list(dict_fips.keys()):
                print('State: ' + state)
                for year in tqdm(years_to_import):
                    try:
                        list_df_years.append(
                            query_census(df_urls      = df_urls
                                          , api_key   = api_key
                                          , estimate  = estimate
                                          , sample    = sample_type
                                          , geography = geography
                                          , variables = variables
                                          , year      = year
                                          , state     = state
                                          , county    = dict_fips[state])
                        )
                    except Exception as e: print(e)
            df_years = pd.concat(list_df_years)
                        
        if import_tab == 'MSA':
            for year in tqdm(years_to_import):
                try:
                    list_df_years.append(
                        query_census(df_urls      = df_urls
                                      , api_key   = api_key
                                      , estimate  = estimate
                                      , sample    = sample_type
                                      , geography = geography
                                      , variables = variables
                                      , year      = year
                                      , msa       = msa_to_import)
                    )
                except Exception as e: print(e)
            df_years = pd.concat(list_df_years)
        list_df_vars.append(df_years)
    
    if geography == 'Tracts':
        df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'tract', 'Year'], how = 'outer'), list_df_vars)
    if geography == 'Counties':
        df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'Year'], how = 'outer'), list_df_vars)
    if geography == 'MSA':
        df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year'], how = 'outer'), list_df_vars)
    list_df_census.append(df_vars_all)

    print("")
    print("Reducing all tables together into one final table...")
    print("")
    
    if geography == 'Tracts':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'tract', 'Year']), list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'tract', 'Year']).reset_index()
    if geography == 'Counties':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'Year']), list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'Year']).reset_index()
    if geography == 'MSA':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']), list_df_census)
        df_census_raw = df_census_raw.set_index(['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']).reset_index()



## For DEC tables
if estimate == 'DEC':

    print("Importing and compiling Decennial data from the Census Bureau...")
    print("")

    # initialize empty list to store data frames
    # import multiple years and counties
    # iterate through each table and import all variables needed from each table
    # combine all years and counties
    # reduce all tables/variables pulled into one table
    
    list_df_census = []
    
    for state in list(dict_fips.keys()):
        print('State: ' + state)
        for year in tqdm(years_to_import):
            try:
                list_df_census.append(
                    query_census(df_urls       = df_urls
                                   , api_key   = api_key
                                   , estimate  = estimate
                                   , sample    = sample_type
                                   , geography = geography
                                   , variables = ','.join(dict_vars[str(year)])
                                   , year      = year
                                   , state     = state
                                   , county    = dict_fips[state])
                )
            except Exception as e: print(e)
                    
    if geography == 'Tracts':
        df_census_raw = pd.concat(list_df_census)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'tract', 'Year']).reset_index()
    if geography == 'Counties':
        df_census_raw = pd.concat(list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'Year']).reset_index()


## For PUMS tables
if geography == 'PUMA':
    
    print("Importing and compiling PUMS data from the Census Bureau...")
    print("")
    
    # initialize empty list to store data frames
    # import multiple years and PUMAs
    # import all variables
    # combine all years and PUMAs
    # outer join variables onto ID fields for each geography type
    # calculate margin of error using replicate weights
    
    for state in list(dict_fips.keys()):
        print('State: ' + state)
        list_df_years = []
        for year in years_to_import:
            print("")
            print('Year: ' + str(year))
            list_df_vars = []
            try:
                list_table_vars = [dict_vars[str(year)][x:x+45] for x in range(0, len(dict_vars[str(year)]), 45)]
                
                list_variables = []
                for x in list_table_vars:
                    list_variables.append(",".join(x))

                print('Querying variables...')
                for variables in tqdm(list_variables):
                    df_pums = query_census(df_urls       = df_urls
                                             , api_key   = api_key
                                             , estimate  = estimate
                                             , sample    = sample_type
                                             , geography = geography
                                             , variables = 'PUMA,SERIALNO,' + variables # Need to figure out how to parse out groups from variables by year
                                             , year      = year
                                             , state     = state
                                             , puma      = dict_fips[state])
                    df_pums['state'] = state
                    df_pums = df_pums.drop(['public use microdata area'], axis = 1)
                    list_df_vars.append(df_pums)

                # df_vars_years = ft.reduce(lambda left, right: pd.merge(left, right, on = ['state', 'SERIALNO', 'Year', 'PUMA'] + groups + weight, how = 'outer'), list_df_vars)
                df_vars_years = ft.reduce(lambda left, right: pd.merge(left, right, left_index = True, right_index = True, how = 'outer'), list_df_vars)
                # Need to QC that two above lines give the same intended result
                if margin_of_error == 'Yes':
                    df_vars_years = df_vars_years.drop(['state_y', 'SERIALNO_y', 'Year_y', 'PUMA_y'], axis = 1)
                    df_vars_years = df_vars_years.rename(columns = {'state_x':'state', 'SERIALNO_x':'SERIALNO', 'Year_x':'Year', 'PUMA_x':'PUMA'})
                df_vars_years = df_vars_years.set_index(['state', 'SERIALNO', 'Year', 'PUMA']).reset_index()
                df_vars_years.columns = ['state', 'SERIALNO', 'Year', 'PUMA'] + dict_vars[str(np.max(years_to_import))]

                if margin_of_error == 'Yes':
                    print('Calculating margin of error using replicate weights...')
                    cols = [col for col in df_vars_years.columns if weight in col]
                    df_vars_years[cols] = df_vars_years[cols].astype(int)
                    df_me = pd.melt(df_vars_years
                                     , id_vars    = ['state', 'SERIALNO', 'Year', 'PUMA'] + groups + [weight]
                                     , var_name   = 'replicates'
                                     , value_name = 'replicate_weights')
                    df_me['sq_diff'] = (df_me['replicate_weights'] - df_me[weight])**2
                    df_me = df_me.groupby(['state', 'SERIALNO', 'Year', 'PUMA'] + groups + [weight], as_index = False)['sq_diff'].agg(sum)
                    df_me['variance'] = df_me['sq_diff']*(4/80)
                    df_me['SE'] = np.sqrt(df_me['variance'])
                    df_me['ME'] = df_me['SE']*1.645
                    df_me = df_me[['state', 'SERIALNO', 'Year', 'PUMA'] + groups + [weight, 'ME']].drop_duplicates()
                    cols_to_drop = df_vars_years.columns[df_vars_years.columns.str.contains(weight)]
                    df_vars_years = df_vars_years.drop(cols_to_drop[:-1], axis = 1)
                    df_vars_years = df_vars_years.drop_duplicates()
                    df_vars_years = df_vars_years.merge(df_me, on = ['state', 'SERIALNO', 'Year', 'PUMA'] + groups + [weight], how = 'left')
                    df_vars_years = df_vars_years.drop_duplicates()
                list_df_years.append(df_vars_years)
                print('Success!')
                
            except Exception as e: print(e)
                                
    df_census_raw = pd.concat(list_df_years)


## For CPS tables
if estimate == 'CPS':

    
    print("Importing and compiling CPS data from the Census Bureau...")
    print("")
    
    list_df_census = []


    for state in list(dict_fips.keys()):
        print('State: ' + state)
        for year in tqdm(years_to_import):
            try:
                temp = query_census(df_urls      = df_urls
                                     , api_key   = api_key
                                     , estimate  = estimate
                                     , sample    = sample_type
                                     , geography = geography
                                     , variables = 'HRHHID,HRHHID2,'+','.join(dict_vars[str(year)])
                                     # , variables = ','.join(dict_vars[str(year)])
                                     , year      = year
                                     , state     = state
                                     , county    = dict_fips[state])
                list_df_census.append(temp)
            except Exception as e: print(e)
                
    df_census_raw = pd.concat(list_df_census)

    # merge county name onto table
    df_census_raw['state' ] = df_census_raw['state' ].astype(str).apply('{:0>2}'.format)
    df_census_raw['county'] = df_census_raw['county'].astype(str).apply('{:0>3}'.format)
    df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                          , left_on = ['state', 'county']
                                          , right_on = ['State FIPS', 'County FIPS'])
    df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
    df_census_raw = df_census_raw.set_index(['state', 'county', 'County Name', 'Year']).reset_index()
    

print("")
print("Finished!! (づ ◕‿◕ )づ")
print(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---")

Importing and compiling PUMS data from the Census Bureau...

State: 06

Year: 2016
Querying variables...


100%|██████████| 1/1 [00:08<00:00,  8.71s/it]


"['state_y', 'SERIALNO_y', 'Year_y', 'PUMA_y'] not found in axis"

Year: 2017
Querying variables...


100%|██████████| 1/1 [00:08<00:00,  8.77s/it]


"['state_y', 'SERIALNO_y', 'Year_y', 'PUMA_y'] not found in axis"

Year: 2018
Querying variables...


...

100%|██████████| 1/1 [00:10<00:00, 10.31s/it]


"['state_y', 'SERIALNO_y', 'Year_y', 'PUMA_y'] not found in axis"

Year: 2022
Querying variables...


  0%|          | 0/1 [00:00<?, ?it/s]

invalid syntax (<unknown>, line 1)


ValueError: No objects to concatenate

In [53]:
# view raw data
pd.set_option('display.max_columns', None)
print(df_census_raw.shape)
print(df_census_raw.Year.unique())
df_census_raw.head(3)

(9220, 14)
[2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022]


,state,county,County Name,Year,HRHHID,HRHHID2,HES1A,HES1B,HES1C,HES1D,PEHSPNON,PERRP,PTDTRACE,HHSUPWGT
0,06,061,Placer,2009,500164009202650,85002,1,2,1,2,2,1,1,4058.9581
1,06,061,Placer,2009,500164009202650,85002,1,2,1,2,2,4,1,4058.9581
2,06,061,Placer,2009,527091689600968,85001,2,2,1,1,2,2,1,3107.7496


***

## Processing

***

In [54]:
## Make copy of data frame
df_census = df_census_raw.copy()

# Replace weird missing values with np.nan
# Melt data from wide to long
# Convert imported values to numeric
# Merge cleam label field, variable mapping, race/ethnicity, and sorting field
# Remove unneeded columns
# Manually check column names and clean as needed
# Adjust dollars for inflation, if needed
if sample_type in ['ACS', 'SUBJECT']:
    print('Processing 1...')
    df_census = acs_processing_1(df_census, df_vars, indicator_name, geography)
    display(df_census.head(3))

# Reorganize margin of error fields
# Create "Categorical" race/ethnicity field for sorting
# Sort by geography, variable mapping, and race/ethnicity
# sort and then remove categorical field
if sample_type in ['ACS', 'SUBJECT']:
    print('Processing 2...')
    df_census = acs_processing_2(df_census, margin_of_error)
    display(df_census.head(3))

# Final processing step for ACS data
# Link various FIPS codes
# Roll up population/households/SE's to the desired geography and variable groupings
# Calculate percentages by geography, race/ethnicity, and variables
if sample_type in ['ACS', 'SUBJECT']:
    print('Processing 3...')
    if geography == 'Tracts':
        df_tracts1, df_tracts2 = acs_processing_3(df_census, indicator_name, geography, percentages, margin_of_error, MOE_thresh, df_fips)
        display(df_tracts1.head(3), df_tracts2.head(3))
    if geography == 'Counties':
        df_counties1, df_counties2, df_mpo1, df_mpo2 = acs_processing_3(df_census, indicator_name, geography, percentages, margin_of_error, MOE_thresh, df_fips)
        display(df_mpo1.head(3), df_mpo2.head(3))
    if geography == 'MSA':
        df_msa1, df_msa2 = acs_processing_3(df_census, indicator_name, geography, percentages, margin_of_error, MOE_thresh)
        display(df_msa1.head(3), df_msa2.head(3))


if sample_type in ['PUMS', 'FOODSEC']:
    # Clean missing values, standardize how the categories are assigned by number, standardize state FIPS code
    # Convert weighted column to integer, convert value fields to string to use as merge field
    # Reshape data dictionary of values/descriptions and reorganize columns
    # Merge meaningful value descriptions onto imported data

    if sample_type == 'PUMS':
        df_census['PUMA'] = df_census['PUMA'].astype(str).apply('{:0>5}'.format)
        
    if (table_type == 'H') & ('SPORDER' in df_census.columns):
        df_census = df_census[df_census['SPORDER'] == '1']
        df_census = df_census.drop('SPORDER', axis = 1)

    if sample_type == 'FOODSEC':
        cols = ['state', 'county', 'County Name', 'Year'] + dict_vars[str(year)]
        df_census = df_census[cols]

    for group in groups2:
        df_census[group] = df_census[group].astype(str).apply('{:0>2}'.format)
        
    df_vars   ['Value1'] = df_vars   ['Value1'].astype(str).apply('{:0>2}'.format)
    df_census ['state' ] = df_census ['state' ].astype(str).apply('{:0>2}'.format)

    df_census.loc[df_census[weight] == '' , weight] = 0
    df_census.loc[df_census[weight] == '-', weight] = 0
    if sample_type == 'PUMS':
        df_census[weight] = df_census[weight].astype(int)
    df_census[groups2] = df_census[groups2].astype("string")

    df_vars2 = df_vars.pivot_table(index = ['Year', 'Value1']
                                           , columns = 'ID2'
                                           , values = 'Description2'
                                           , aggfunc = lambda x: x).reset_index()
    cols = ['Year', 'Value1'] + groups2
    df_vars2 = df_vars2[cols]

    list_values = []
    for group in groups2:
        list_values = list_values + list(df_census[group].values)
    set_values = set(list_values)
    
    df_vars2 = df_vars2[df_vars2['Value1'].isin(set_values)]
    df_vars2 = df_vars2.add_suffix('_desc').rename(columns = {'Value1_desc':'Value1', 'Year_desc':'Year'})

    for col in cols[2:]:
        df_census = df_census.merge(df_vars2[['Value1', col+'_desc', 'Year']], left_on = [col, 'Year'], right_on = ['Value1', 'Year'], how = 'inner')
        df_census[col] = df_census[col+'_desc']
        df_census = df_census.drop(['Value1', col+'_desc'], axis = 1)

    # if sample_type == 'FOODSEC':
    #     df_census.loc[df_census['PRDTHSP_desc'].str.contains(','), 'PRDTHSP_desc'] = 'Not Hispanic or Latino'
    display(df_census.head(3))


# Remove rows with missing values
# Sort by PUMA, Year, then by each group
# Only keep description mappings, remove the original PUMS values
# Rollup using suggested weight field
# TODO: merge on PUMA name field
if sample_type == 'PUMS':
    df_census = df_census.dropna()
    
    df_fips_pums = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'PUMAcodes'
                            , dtype = {'STATEFP': object, 'COUNTYFP': object, 'TRACTCE': object, 'PUMA5CE': object})
    df_fips_pums = df_fips_pums[df_fips_pums['STATEFP'].isin(list(dict_fips.keys()))]
    df_fips_pums = df_fips_pums[['STATEFP', 'PUMA5CE', 'PUMA NAME', 'COUNTYFP', 'Years']].rename(columns = {'PUMA5CE':'PUMA', 'STATEFP':'state'}).drop_duplicates()

    df_census1 = df_census[df_census['Year'].isin(sequence(2012, 2021, 1))]
    df_census2 = df_census[df_census['Year'].isin(sequence(2022, 2031, 1))]
    
    df_census1 = df_census1.merge(df_fips_pums[df_fips_pums['Years'] == '2012-2021'], on = ['state', 'PUMA'], how = 'left')
    df_census2 = df_census2.merge(df_fips_pums[df_fips_pums['Years'] == '2022-2031'], on = ['state', 'PUMA'], how = 'left')
    df_census = pd.concat([df_census1, df_census2])
    df_census = df_census.drop('Years', axis = 1)

    df_census['COUNTYFP'] = df_census['COUNTYFP'].astype(str).apply('{:0>3}'.format)
    df_census = df_census.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                          , left_on = ['state', 'COUNTYFP']
                                          , right_on = ['State FIPS', 'County FIPS'])
    df_census.drop(['state', 'COUNTYFP'], axis = 1, inplace = True)
    
    df_census = df_census.set_index(['State FIPS', 'County FIPS', 'County Name', 'Year']).reset_index()
    
    df_census = df_census.sort_values(['PUMA', 'Year'] + groups, ascending = [True, False] + [item in groups for item in groups])
    
    if 'HISP' in groups:
        df_census.loc[df_census['HISP'] == 'Hispanic or Latino', 'RAC1P'] = 'Hispanic or Latino'
        df_census = df_census.drop('HISP', axis = 1)
        groups.remove('HISP')

    if 'HHLDRHISP' in groups:
        df_census.loc[df_census['HHLDRHISP'] == 'Hispanic or Latino', 'HHLDRRAC1P'] = 'Hispanic or Latino'
        df_census = df_census.drop('HHLDRHISP', axis = 1)

    df_puma     = df_census.drop([                     'County FIPS', 'County Name', 'SERIALNO'], axis = 1)                     
    df_counties = df_census.drop(['PUMA', 'PUMA NAME'                              , 'SERIALNO'], axis = 1)
    df_mpo      = df_census.drop(['PUMA', 'PUMA NAME', 'County FIPS', 'County Name', 'SERIALNO'], axis = 1)

    if margin_of_error == 'Yes':
        df_puma     = df_puma    .groupby(list(df_puma    .drop([weight, 'ME'], axis = 1).columns), as_index = False, sort = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
        df_counties = df_counties.groupby(list(df_counties.drop([weight, 'ME'], axis = 1).columns), as_index = False, sort = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
        df_mpo      = df_mpo     .groupby(list(df_mpo     .drop([weight, 'ME'], axis = 1).columns), as_index = False, sort = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
    if margin_of_error == 'No':
        df_puma     = df_puma    .groupby(list(df_puma    .drop([weight      ], axis = 1).columns), as_index = False, sort = False).agg(Total = (weight, 'sum'))
        df_counties = df_counties.groupby(list(df_counties.drop([weight      ], axis = 1).columns), as_index = False, sort = False).agg(Total = (weight, 'sum'))
        df_mpo      = df_mpo     .groupby(list(df_mpo     .drop([weight      ], axis = 1).columns), as_index = False, sort = False).agg(Total = (weight, 'sum'))

    if percentages == 'Yes':
        if margin_of_error == 'Yes':
            df_puma    ['Percentage'] = 100*df_puma    ['Total'] / df_puma    .groupby(list(df_puma    .drop(groups[:-1] + ['Total', 'ME'], axis = 1).columns))['Total'].transform('sum')
            df_counties['Percentage'] = 100*df_counties['Total'] / df_counties.groupby(list(df_counties.drop(groups[:-1] + ['Total', 'ME'], axis = 1).columns))['Total'].transform('sum')
            df_mpo     ['Percentage'] = 100*df_mpo     ['Total'] / df_mpo     .groupby(list(df_mpo     .drop(groups[:-1] + ['Total', 'ME'], axis = 1).columns))['Total'].transform('sum')
        if margin_of_error == 'No':
            df_puma    ['Percentage'] = 100*df_puma    ['Total'] / df_puma    .groupby(list(df_puma    .drop(groups[:-1] + ['Total'      ], axis = 1).columns))['Total'].transform('sum')
            df_counties['Percentage'] = 100*df_counties['Total'] / df_counties.groupby(list(df_counties.drop(groups[:-1] + ['Total'      ], axis = 1).columns))['Total'].transform('sum')
            df_mpo     ['Percentage'] = 100*df_mpo     ['Total'] / df_mpo     .groupby(list(df_mpo     .drop(groups[:-1] + ['Total'      ], axis = 1).columns))['Total'].transform('sum')

    df_puma     = df_puma    .set_index(['State FIPS', 'PUMA'       , 'PUMA NAME'  ]).reset_index()
    df_counties = df_counties.set_index(['State FIPS', 'County FIPS', 'County Name']).reset_index()
    df_mpo['MPO'] = 'SACOG'
    df_mpo = df_mpo.set_index(['State FIPS', 'MPO']).reset_index()

    display(df_puma.head(3), df_counties.head(3), df_mpo.head(3))
    

KeyError: 'PUMA'

In [57]:
if sample_type in ['PUMS', 'FOODSEC']:
    # Clean missing values, standardize how the categories are assigned by number, standardize state FIPS code
    # Convert weighted column to integer, convert value fields to string to use as merge field
    # Reshape data dictionary of values/descriptions and reorganize columns
    # Merge meaningful value descriptions onto imported data

    if sample_type == 'PUMS':
        df_census['PUMA'] = df_census['PUMA'].astype(str).apply('{:0>5}'.format)
        
    if (table_type == 'H') & ('SPORDER' in df_census.columns):
        df_census = df_census[df_census['SPORDER'] == '1']
        df_census = df_census.drop('SPORDER', axis=1)

    if sample_type == 'FOODSEC':
        cols = ['state', 'county', 'County Name', 'Year'] + dict_vars[str(year)]
        df_census = df_census[cols]

    for group in groups2:
        if group in df_census.columns:
            df_census[group] = df_census[group].astype(str).apply('{:0>2}'.format)
        else:
            print(f"Column {group} not found in df_census")

    if 'Value1' in df_vars.columns:
        df_vars['Value1'] = df_vars['Value1'].astype(str).apply('{:0>2}'.format)
    else:
        print("Column Value1 not found in df_vars")

    if 'state' in df_census.columns:
        df_census['state'] = df_census['state'].astype(str).apply('{:0>2}'.format)
    else:
        print("Column state not found in df_census")

    if weight in df_census.columns:
        df_census.loc[df_census[weight] == '', weight] = 0
        df_census.loc[df_census[weight] == '-', weight] = 0
        if sample_type == 'PUMS':
            df_census[weight] = df_census[weight].astype(int)
    else:
        print(f"Column {weight} not found in df_census")

    df_census[groups2] = df_census[groups2].astype("string")

    df_vars2 = df_vars.pivot_table(index=['Year', 'Value1'], columns='ID2', values='Description2', aggfunc=lambda x: x).reset_index()
    cols = ['Year', 'Value1'] + groups2
    df_vars2 = df_vars2[cols]

    list_values = []
    for group in groups2:
        if group in df_census.columns:
            list_values += list(df_census[group].values)
        else:
            print(f"Column {group} not found in df_census")

    set_values = set(list_values)
    
    df_vars2 = df_vars2[df_vars2['Value1'].isin(set_values)]
    df_vars2 = df_vars2.add_suffix('_desc').rename(columns={'Value1_desc': 'Value1', 'Year_desc': 'Year'})

    for col in cols[2:]:
        if col in df_census.columns and f"{col}_desc" in df_vars2.columns:
            df_census = df_census.merge(df_vars2[['Value1', f"{col}_desc", 'Year']], left_on=[col, 'Year'], right_on=['Value1', 'Year'], how='inner')
            df_census[col] = df_census[f"{col}_desc"]
            df_census = df_census.drop(['Value1', f"{col}_desc"], axis=1)
        else:
            print(f"Columns {col} or {col}_desc not found during merging")

    # if sample_type == 'FOODSEC':
    #     df_census.loc[df_census['PRDTHSP_desc'].str.contains(','), 'PRDTHSP_desc'] = 'Not Hispanic or Latino'
    display(df_census.head(3))

KeyError: 'PUMA'

In [ ]:
# unique(list(df_census.PRDTHSP_desc.values))
# df_census['PTDTRACE'].unique()

# df_census.loc[df_census['PRDTHSP'] == '-1', 'PRDTHSP_desc'] = 'Not Hispanic or Latino'
# df_census['PRDTHSP_desc'].unique()

# df_census[df_census['PRDTHSP'] == '01']

In [ ]:
# df_census = df_census_raw.copy()
# cols = ['state', 'county', 'County Name', 'Year'] + dict_vars[str(year)]
# df_census = df_census[cols]

# ## For non-integers only
# for group in groups:
#     df_census[group] = df_census[group].astype(str).apply('{:0>2}'.format)
    
# df_vars['Value1'] = df_vars['Value1'].astype(str).apply('{:0>2}'.format)
# df_census ['state' ] = df_census ['state' ].astype(str).apply('{:0>2}'.format)
# df_census.loc[df_census[weight] == '', weight] = 0
# df_census[weight] = df_census[weight].astype(int)
# df_census[groups] = df_census[groups].astype("string")
# df_vars2 = df_vars.pivot_table(index = ['Year', 'Value1']
#                                        , columns = 'ID2'
#                                        , values = 'Description2'
#                                        , aggfunc = lambda x: x).reset_index()
# cols = ['Year', 'Value1'] + groups
# df_vars2 = df_vars2[cols]

In [ ]:
# Final renaming of tables for cleanliness
if geography == 'Tracts':
    df_tracts1 = rename_census(df_tracts1        = df_tracts1
                               , geography       = geography
                               , indicator_name  = indicator_name
                               , margin_of_error = margin_of_error)
    display(df_tracts1.head(3))   
if geography == 'Counties':
    df_counties1, df_mpo1 = rename_census(df_counties1      = df_counties1
                                          , df_mpo1         = df_mpo1
                                          , geography       = geography
                                          , indicator_name  = indicator_name
                                          , margin_of_error = margin_of_error)
    display(df_counties1.head(3), df_mpo1.head(3))   
if geography == 'MSA':
    df_msa1 = rename_census(df_msa1           = df_msa1
                            , geography       = geography
                            , indicator_name  = indicator_name
                            , margin_of_error = margin_of_error)
    display(df_msa1.head(3))
if geography == 'PUMA':
    df_puma, df_counties, df_mpo = rename_census(df_puma           = df_puma
                                                 , df_counties     = df_counties
                                                 , df_mpo          = df_mpo
                                                 , geography       = geography
                                                 , indicator_name  = indicator_name
                                                 , margin_of_error = margin_of_error)
    display(df_puma.head(3), df_counties.head(3), df_mpo.head(3))

***

## Exporting

***

In [ ]:
# Set file path for exporting
path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out, indicator_name + ' ' + folder)
path_out_csv  = os.path.join(path_agol, indicator_name)

print('Excel files exported here: ' + path_out_xlsx)
print('CSV files exported here: '   + path_out_csv )

if geography == 'Tracts':
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Tracts '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_tracts1.to_excel(writer, index = False, sheet_name = 'Tracts')
        # df_tracts2.to_excel(writer, index = False, sheet_name = 'Tracts wide')

if geography == 'Counties':
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Counties '+estimate+'.xlsx'), engine='xlsxwriter') as writer:
    with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Counties '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_counties1.to_excel(writer, index = False, sheet_name = 'Counties')
        # df_counties2.to_excel(writer, index = False, sheet_name = 'Counties wide')
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MPO '+estimate+'.xlsx'), engine='xlsxwriter') as writer:
    with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MPO '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_mpo1.to_excel(writer, index = False, sheet_name = 'MPO')
        # df_mpo2.to_excel(writer, index = False, sheet_name = 'MPO wide')
             

if geography == 'MSA':
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MSA '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_msa1.to_excel(writer, index = False, sheet_name = 'MSA')
        # df_msa2.to_excel(writer, index = False, sheet_name = 'MSA wide')


if geography == 'PUMA':

    # df_census.columns = [re.sub('_desc', '', col) for col in df_census.columns]
    # df_census_csv = df_census.rename(columns = {'Year':'year', 'state': 'State FIPS'})

    # df_census_csv.to_csv(os.path.join(path_out_csv, name_output_PUMA_csv), index = False)

    name_output_xlsx = re.sub('ACS', 'PUMS', name_output_xlsx)

    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
        df_puma.to_excel(writer, index = False, sheet_name = 'PUMA')
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
        df_county.to_excel(writer, index = False, sheet_name = 'Counties')
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
        df_mpo.to_excel(writer, index = False, sheet_name = 'MPO')


print('')
print("Successfully exported")

## Testing the SE properties

Couple of resources for proving the relationship between SE and MSE. 

https://www.formpl.us/blog/margin-of-error#:~:text=A%20margin%20of%20error%20is%20a%20statistical%20measure%20that%20accounts,deviation%20of%20the%20data%20set.

https://www.wallstreetmojo.com/margin-of-error-formula/

https://www.inchcalculator.com/margin-of-error-calculator/

The margin of error is given by the following equation.

$$
ME = Z * \frac{\sigma}{n}
$$

Where the quantity $\frac{\sigma}{n}$ is the standard error $(SE)$. 

Now, let us consider the sum of squares for these quantities. 

$$
ME_{\text{total}} = \sqrt{\sum^n_{i=1} (ME_{i}^{2})}, \ \ \ SE_{\text{total}} = \sqrt{\sum^n_{i=1} (SE_{i}^{2})}
$$

Applying the prior definitions, we arrive at the following. 

$$
ME_{\text{total}} = \sqrt{\sum^n_{i=1} (Z * SE_i)^{2}}
$$

$$
= \sqrt{\sum^n_{i=1} (Z^2 * SE_i^2)}
$$

$$
= Z * \sqrt{\sum^n_{i=1} SE_i^2}
$$

$$
= 1.645 * SE_{\text{total}}
$$

Thus, we can confirm that the relationship between the square root of the sum of squares using the margin of error is equal to the square root of the sum of squares using the standard error, multiplied by the $Z$ score 1.645. 